In [4]:
print("Activated")

Activated


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim


from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split, Subset

### 1. Transformation setup

In [6]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

### 2. Load train and test dataset

In [7]:
train_dataset_path = r"C:\Users\TAQICOMPUTERS\Desktop\brain-tumor-classifier\Training"
test_dataset_path = r"C:\Users\TAQICOMPUTERS\Desktop\brain-tumor-classifier\Testing"

train_dataset = ImageFolder(
    train_dataset_path,
    transform= transform
)

test_dataset = ImageFolder(
    test_dataset_path,
    transform= transform
)

# train classes
print(train_dataset.classes)

['glioma', 'meningioma', 'notumor', 'pituitary']


### 3. Train / Validation Split

In [8]:
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_data, val_data = random_split(
    train_dataset,
    [train_size, val_size]
)

### 3. DataLoaders

In [9]:
train_loader = DataLoader(
    train_data,
    batch_size=74,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=74,
    shuffle=False
)

val_loader = DataLoader(
    val_data,
    batch_size=74,
    shuffle=False

)

### 4. Model Architecture

In [10]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(16 * 16 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 4)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

### 5. Build Model

In [11]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### 6. Training Loop

In [12]:
epochs = 8
model.train()

print("--- Training Started ---")
for epoch in range(epochs):
    epoch_train_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()      
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item()
    
    avg_loss = epoch_train_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} - Loss: {avg_loss:.4f}")


--- Training Started ---
Epoch 1/8 - Loss: 0.8531
Epoch 2/8 - Loss: 0.4634
Epoch 3/8 - Loss: 0.3103
Epoch 4/8 - Loss: 0.2170
Epoch 5/8 - Loss: 0.1532
Epoch 6/8 - Loss: 0.0958
Epoch 7/8 - Loss: 0.0578
Epoch 8/8 - Loss: 0.0356


### Evaluation

In [13]:
model.eval() 
correct_labels = 0
total_labels = 0

with torch.no_grad(): 
    for images, labels in val_loader:
        outputs = model(images)                
        _, predicted = torch.max(outputs, 1)   
        correct_labels += (predicted == labels).sum().item()  
        total_labels += labels.size(0)                        

accuracy = (correct_labels / total_labels) * 100
print(f"\nValidation Accuracy: {accuracy:.2f}%")

torch.save(model.state_dict(), 'brain_tumor_classifier.pth')
print("Model saved successfully as 'brain_tumor_classifier.pth'!")


Validation Accuracy: 90.71%
Model saved successfully as 'brain_tumor_classifier.pth'!
